In [1]:
import polars as pl
import json

def lookup_person(file_path, target_id, id_column="id"):
    try:
        # Filter lazily and collect only the relevant row
        df = pl.scan_csv(file_path).filter(pl.col(id_column) == target_id).collect()
        
        if df.height > 0:
            print(f"--- Records Found: {df.height} match(es) for ID '{target_id}' in {file_path} ---")
            
            records = df.to_dicts()
            
            for i, record in enumerate(records, 1):
                print(f"Result {i}:")
                
                # Pre-process the record to parse any JSON strings found in fields
                processed_record = {}
                for key, value in record.items():
                    if isinstance(value, str):
                        # Clean up common Polars/CSV artifacts before parsing
                        stripped_val = value.strip()
                        if (stripped_val.startswith('[') and stripped_val.endswith(']')) or \
                           (stripped_val.startswith('{') and stripped_val.endswith('}')):
                            try:
                                processed_record[key] = json.loads(value)
                            except json.JSONDecodeError:
                                processed_record[key] = value
                        else:
                            processed_record[key] = value
                    else:
                        processed_record[key] = value

                # Print the fully nested structure with indentation
                print(json.dumps(processed_record, indent=4, ensure_ascii=False))
                print("." * 20)
                
            print("-" * 30)
        else:
            print(f"Status: No matches found for ID '{target_id}' in {file_path}.")
            
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' does not exist.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# --- Configuration & Execution ---
ID_TO_FIND = "melissa-jardine-phd-b8230b21"
COLUMN_NAME = "id"
files = ["../Data/nam_processed_data.csv", "../Data/final.csv"]

for file in files:
    lookup_person(file, ID_TO_FIND, COLUMN_NAME)
    print("\n" + "="*40 + "\n")

--- Records Found: 1 match(es) for ID 'melissa-jardine-phd-b8230b21' in ../Data/nam_processed_data.csv ---
Result 1:
{
    "id": "melissa-jardine-phd-b8230b21",
    "name": "Melissa Jardine PhD",
    "city": "Thành Phố Hội An, Quang Nam, Vietnam",
    "country_code": "VN",
    "position": "Policing & Security | Public Health | Gender Advisor",
    "about": "I am an international policing advisor undertaking research and capacity building activities across the globe, with current projects in the Pacific, Central Asia, South and Southeast Asia and Africa. Author of two books: – Jardine, M. (2022). Policing in a Changing Vietnam: Towards a Global Account of Policing. Routledge. – Jardine, M. (2022). Women, Police and Security: Reforming the Afghan National Police. Global Law Enforcement and Public Health Association. I have a long term interest in the development of policing and security in Asia. I have conducted primary research on provincial economic growth in China and its relationship